In [5]:
# Setup
import sys
import json
import inspect

# Add the dataset folder to Python's module search path
sys.path.append('/kaggle/input/datasets/noorulainakagglehehe/lab09-taskmaterial')

import langchain
import langchain_core
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from typing import Literal

import lab9_npc as npc
from lab9_npc import llm, set_seed, check_inventory, roll_dice

print("langchain-core", langchain_core.__version__, "| langchain", langchain.__version__)

langchain-core 1.7.0 | langchain 1.4.7


In [6]:
# Activity 1 - Call the model and log the call
npc.LOG.clear()
set_seed(0)
reply = llm.invoke("Say hello in one short sentence.")
print(type(reply).__name__, "|", reply.content)
print("calls logged:", len(npc.LOG))


AIMessage | Hello from the Lab 9 stand-in. I am a Python function, not a language model.
calls logged: 1


In [8]:
# Activity 2 - The hand-rolled NPC
SYSTEM = "You are {npc}, a {role}. Mood: {mood}. Stay in character and answer in one or two sentences."
 
def hand_npc(player_text, name="Brom", role="gruff blacksmith", mood="grumpy"):
    system = SYSTEM.format(npc=name, role=role, mood=mood)
    messages = [SystemMessage(content=system), HumanMessage(content=player_text)]
    return llm.invoke(messages).content.strip()
 
for line in ["Hello there.", "Do you have any work for me?"]:
    print("Player:", line)
    print("  Brom:", hand_npc(line))


Player: Hello there.
  Brom: *grunts* I am Brom the gruff blacksmith. What do you need?
Player: Do you have any work for me?
  Brom: *grunts* Wolves are raiding the mill. Bring me three pelts.


In [9]:
# Activity 3 - The same NPC as a LangChain chain
prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "{player}")])
npc_chain = prompt | llm | StrOutputParser()
 
BROM = {"npc": "Brom", "role": "gruff blacksmith", "mood": "grumpy"}
for line in ["Hello there.", "Do you have any work for me?"]:
    print("Player:", line)
    print("  Brom:", npc_chain.invoke({**BROM, "player": line}))
print("same answers as the hand-rolled version:",
      npc_chain.invoke({**BROM, "player": "Hello there."}) == hand_npc("Hello there."))


Player: Hello there.
  Brom: *grunts* I am Brom the gruff blacksmith. What do you need?
Player: Do you have any work for me?
  Brom: *grunts* Wolves are raiding the mill. Bring me three pelts.
same answers as the hand-rolled version: True


In [10]:
# Activity 4 - Batch: three player lines in one call
lines = ["Hello there.", "Do you have any work for me?", "My name is Aria."]
npc.LOG.clear()
answers = npc_chain.batch([{**BROM, "player": x} for x in lines])
for x, a in zip(lines, answers):
    print(f"{x:30} -> {a}")
print("model calls:", len(npc.LOG))


Hello there.                   -> *grunts* I am Brom the gruff blacksmith. What do you need?
Do you have any work for me?   -> *grunts* Wolves are raiding the mill. Bring me three pelts.
My name is Aria.               -> *grunts* Well met, Aria. I am Brom the gruff blacksmith.
model calls: 3


In [11]:
#TASK 01

PERSONAS = [
    {"npc": "Brom", "role": "gruff blacksmith", "mood": "grumpy"},
    {"npc": "Lyra", "role": "village healer", "mood": "cheerful"},
    {"npc": "Pip", "role": "shy messenger", "mood": "nervous"},
]

# TODO 1: loop over PERSONAS, ask each NPC "Do you have any work for me?"
#         with npc_chain and print one line per NPC as shown below.
question = "Do you have any work for me?"

for p in PERSONAS:
    reply = npc_chain.invoke({**p, "player": question})
    print(f"{p['npc']:<5} ({p['mood']:<8}): {reply}")

print("-" * 40)

# TODO 2: add a fourth persona of your own (mood: grumpy, cheerful or nervous)
#         and run it too.
PERSONAS.append({"npc": "Mira", "role": "tavern keeper", "mood": "cheerful"})

for p in PERSONAS:
    reply = npc_chain.invoke({**p, "player": question})
    print(f"{p['npc']:<5} ({p['mood']:<8}): {reply}")

Brom  (grumpy  ): *grunts* Wolves are raiding the mill. Bring me three pelts.
Lyra  (cheerful): Ha! Wolves are raiding the mill. Bring me three pelts.
Pip   (nervous ): Um... Wolves are raiding the mill. Bring me three pelts.
----------------------------------------
Brom  (grumpy  ): *grunts* Wolves are raiding the mill. Bring me three pelts.
Lyra  (cheerful): Ha! Wolves are raiding the mill. Bring me three pelts.
Pip   (nervous ): Um... Wolves are raiding the mill. Bring me three pelts.
Mira  (cheerful): Ha! Wolves are raiding the mill. Bring me three pelts.


In [12]:
#TASK 02

class NPCReply(BaseModel):
    say: str
    emotion: Literal["angry", "happy", "afraid", "neutral"]
    action: Literal["none", "offer_quest", "open_shop"]
 
parser = PydanticOutputParser(pydantic_object=NPCReply)
json_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + "\n{format_instructions}"), ("human", "{player}")
]).partial(format_instructions=parser.get_format_instructions())
 
# TODO 1: json_chain = prompt | model | parser
json_chain = json_prompt | llm | parser

# TODO 2: safe_chain = json_chain wrapped with .with_retry(...):
#         retry only on OutputParserException, at most 3 attempts, wait_exponential_jitter=False
safe_chain = json_chain.with_retry(
    retry_if_exception_type=(OutputParserException,),
    stop_after_attempt=3,
    wait_exponential_jitter=False
)
 
def measure(chain, seeds=range(20)):
    # TODO 3: for every seed (call set_seed) and every line in `lines` (from Activity 4):
    #         invoke the chain; count usable replies and OutputParserException failures.
    #         Clear npc.LOG first and return (usable, failed, len(npc.LOG)).
    npc.LOG.clear()
    usable = 0
    failed = 0
    for s in seeds:
        set_seed(s)
        for line in lines:
            try:
                chain.invoke({**BROM, "player": line})
                usable += 1
            except OutputParserException:
                failed += 1
    return usable, failed, len(npc.LOG)
 
# TODO 4: for seeds 0 to 3 print "seed N: accepted -> <reply>" or
#         "seed N: REJECTED -> <first line of the error, 60 characters>".
#         Then print the "no retry" and "with retry" lines of the expected output.
for s in range(4):
    set_seed(s)
    try:
        reply = json_chain.invoke({**BROM, "player": lines[1]})
        print(f"seed {s}: accepted -> {reply}")
    except OutputParserException as e:
        first_line = str(e).splitlines()[0][:60]
        print(f"seed {s}: REJECTED -> {first_line}")

for label, chain in [("no retry  ", json_chain), ("with retry", safe_chain)]:
    u, f, calls = measure(chain)
    total = u + f
    pct = round(100 * u / total)
    print(f"{label}: usable {u}/{total} ({pct}%), failed {f}, model calls {calls}")

seed 0: accepted -> say='*grunts* Wolves are raiding the mill. Bring me three pelts.' emotion='angry' action='none'
seed 1: REJECTED -> Invalid json output: Here is the reply: {"say": "*grunts* Wo
seed 2: accepted -> say='*grunts* Wolves are raiding the mill. Bring me three pelts.' emotion='angry' action='none'
seed 3: REJECTED -> Invalid json output: Here is the reply: {"say": "*grunts* Wo
no retry  : usable 41/60 (68%), failed 19, model calls 60
with retry: usable 59/60 (98%), failed 1, model calls 90


In [1]:
#TASK 03
TOOLS = {"check_inventory": check_inventory, "roll_dice": roll_dice}  # [Agent Logic] tool whitelist

def tool_npc(player_text, max_steps=4):
    # [LangChain] message wrappers & prompt formatting
    messages = [SystemMessage(content=SYSTEM.format(**BROM)), HumanMessage(content=player_text)]
    # [LangChain] bind_tools attaches tool schemas to the LLM
    bound = llm.bind_tools(list(TOOLS.values()))
    trace = []  # [Agent Logic] execution trace state
    
    # [Agent Logic] bounded control loop (enforces max_steps)
    for step in range(1, max_steps + 1):
        ai = bound.invoke(messages)  # [Language Model] generates text or tool_calls (invoked via LangChain)
        messages.append(ai)          # [Agent Logic] conversation state management
        
        # TODO case 1: no tool calls in the reply -> return the text and the trace
        if not ai.tool_calls:  # [Agent Logic] stopping condition
            return ai.content, trace + [f"step {step}: final answer"]
        
        # TODO case 2 & 3: validate against whitelist, execute, record trace, append ToolMessage
        for call in ai.tool_calls:  # [Agent Logic] tool dispatch loop
            name = call["name"]
            args = call["args"]
            
            if name not in TOOLS:  # [Agent Logic] whitelist security check
                allowed = ", ".join(TOOLS.keys())
                result = f"unknown tool {name}. Allowed: {allowed}"
            else:
                result = TOOLS[name].invoke(args)  # [LangChain] executes the @tool function
                
            trace.append(f"step {step}: {name}({args}) -> {result}")  # [Agent Logic] logging
            # [LangChain] ToolMessage formats tool output for the model
            messages.append(
                ToolMessage(content=str(result), tool_call_id=call["id"], name=name)
            )
            
    return None, trace + ["stopped: step limit reached"]  # [Agent Logic] fallback termination

NameError: name 'check_inventory' is not defined

In [16]:
#TASK 04
from langchain_core.messages import AIMessage
 
mem_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM), MessagesPlaceholder("history"), ("human", "{player}")])
mem_chain = mem_prompt | llm | StrOutputParser()
HISTORY = {}                       # one list of messages per game session
 
def say(session, text):
    # TODO 1: get the history list for this session (create an empty list if there is none)
    history = HISTORY.setdefault(session, [])
    
    # TODO 2: call mem_chain with {**BROM, "player": text, "history": history}
    reply = mem_chain.invoke({**BROM, "player": text, "history": history})
    
    # TODO 3: append HumanMessage(content=text) and AIMessage(content=reply) to the history
    history.append(HumanMessage(content=text))
    history.append(AIMessage(content=reply))
    
    # TODO 4: return the reply
    return reply
 
for s, text in [("game1", "My name is Aria."), ("game1", "I decided to spare the bandit."),
                ("game1", "What is my name?"), ("game1", "What did I decide?"),
                ("game2", "What is my name?")]:
    print(f"[{s}] Player: {text}\n        Brom: {say(s, text)}")
print("messages stored in game1:", len(HISTORY["game1"]), "| in game2:", len(HISTORY["game2"]))

[game1] Player: My name is Aria.
        Brom: *grunts* Well met, Aria. I am Brom the gruff blacksmith.
[game1] Player: I decided to spare the bandit.
        Brom: *grunts* A bold choice. I will remember it.
[game1] Player: What is my name?
        Brom: *grunts* You told me your name is Aria.
[game1] Player: What did I decide?
        Brom: *grunts* You chose to spare the bandit. I remember.
[game2] Player: What is my name?
        Brom: *grunts* You have not told me your name yet.
messages stored in game1: 8 | in game2: 2


In [17]:
# TASK 05

def needs_tool(text):
    low = text.lower()
    return any(w in low for w in ["buy", "price", "cost", "sword", "potion", "shield", "attack", "roll"])
 
def route(inputs):
    # TODO 1: if needs_tool(inputs["player"]): return a RunnableLambda that
    #         calls tool_npc and returns only the answer.
    #         otherwise: return a RunnableLambda that calls say(session, player).
    if needs_tool(inputs["player"]):
        return RunnableLambda(lambda x: tool_npc(x["player"])[0])
    return RunnableLambda(lambda x: say(x["session"], x["player"]))
 
companion = RunnableLambda(route)
SCRIPT = [("My name is Aria.", "Aria"), ("I chose to guard the village.", "remember"),
          ("How much is the iron sword?", "gold"), ("I attack the wolf!", "rolled"),
          ("What is my name?", "Aria"), ("What did I decide?", "guard the village")]
 
HISTORY.clear(); npc.LOG.clear(); set_seed(5)
# TODO 2: run every line of SCRIPT through
#         companion.invoke({"player": ..., "session": "run1"}).
#         A line PASSES if its check word appears in the reply.
#         Print PASS or FAIL, the line and the reply, and at the end
#         "checks passed: X/6 | model calls: N".
passed = 0
for line, expected in SCRIPT:
    reply = companion.invoke({"player": line, "session": "run1"})
    ok = expected in reply
    passed += ok
    status = "PASS" if ok else "FAIL"
    print(f"{status} | {line:<30} -> {reply}")
print(f"checks passed: {passed}/{len(SCRIPT)} | model calls: {len(npc.LOG)}")
 
# TODO 3: write hand_json(player_text, tries=3): the same job as safe_chain (Task 2)
#         WITHOUT LangChain's parser or with_retry. Build the messages yourself,
#         add the words "Reply with JSON" to the system text, parse with
#         json.loads and NPCReply(**data), retry up to `tries` times,
#         return the NPCReply or None.
def hand_json(player_text, tries=3):
    msgs = [SystemMessage(content=SYSTEM.format(**BROM) + "\nReply with JSON"), HumanMessage(content=player_text)]
    for _ in range(tries):
        try:
            return NPCReply(**json.loads(llm.invoke(msgs).content))
        except Exception:
            pass
    return None

# TODO 4: run hand_json for seeds 0-19 and every line in `lines`, then print
#         "hand-rolled : usable X/60, model calls N, source lines L"
#         (L = len(inspect.getsource(hand_json).splitlines()))
#         and the numbers for safe_chain from measure(safe_chain).
npc.LOG.clear()
hand_usable = 0
for s in range(20):
    set_seed(s)
    for line in lines:
        if hand_json(line) is not None:
            hand_usable += 1
src_lines = len(inspect.getsource(hand_json).splitlines())
print(f"hand-rolled : usable {hand_usable}/60, model calls {len(npc.LOG)}, source lines {src_lines}")

fw_usable, _, fw_calls = measure(safe_chain)
print(f"framework   : usable {fw_usable}/60, model calls {fw_calls}")

PASS | My name is Aria.               -> *grunts* Well met, Aria. I am Brom the gruff blacksmith.
PASS | I chose to guard the village.  -> *grunts* A bold choice. I will remember it.
PASS | How much is the iron sword?    -> *grunts* Yes. iron sword: 2 in stock, 50 gold.
PASS | I attack the wolf!             -> *grunts* You rolled 12. A solid hit!
PASS | What is my name?               -> *grunts* You told me your name is Aria.
PASS | What did I decide?             -> *grunts* You chose to guard the village. I remember.
checks passed: 6/6 | model calls: 8
hand-rolled : usable 59/60, model calls 90, source lines 8
framework   : usable 59/60, model calls 90
